# MLRF — Kaggle training smoke (TV3)

**CHƯA phải kết quả dự đoán DS04 thật.** Mặc định notebook chỉ chạy dữ liệu tổng hợp để kiểm thử đường đi nguồn code → preprocessing → model → metrics → artifact. Chỉ bật chạy dữ liệu được duyệt sau khi Thảo/Sang chốt D10/D12, Đặng chốt D13/D14 và Danh xác nhận quy trình experiment. Không dùng Test để tuning.


In [ ]:
import sys, json, os
from pathlib import Path

USE_APPROVED_DATA = False  # Chỉ đổi sau khi data contract/preprocessing được nghiệm thu
SOURCE_ROOT = None  # Ví dụ Path('/kaggle/input/mlrf-source/src')
DATA_PATH = None    # Ví dụ Path('/kaggle/input/approved-clean-ds04/vehicles_cleaned_v1.parquet')
RUN_ID = 'tv3_smoke_001'

candidates = [Path.cwd() / 'src', Path('/kaggle/input/mlrf-source/src')]
if SOURCE_ROOT is not None:
    candidates.insert(0, Path(SOURCE_ROOT))
source = next((p for p in candidates if (p / 'mlrf' / '__init__.py').exists()), None)
if source is None:
    raise FileNotFoundError('Attach versioned repository source as Kaggle Input or set SOURCE_ROOT to directory containing mlrf/')
sys.path.insert(0, str(source))
print('Model code from:', source)


In [ ]:
from mlrf.training import build_model
from mlrf.testing import split_indices, assert_target_absent
from mlrf.evaluation import regression_metrics
from mlrf.ops import save_run, environment_info
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from mlrf.config import load_project_config

conf = load_project_config()
seed = int(conf['split']['random_seed'])
print('Seed:', seed, 'Versions:', environment_info())


In [ ]:
if USE_APPROVED_DATA:
    if DATA_PATH is None:
        raise ValueError('Set DATA_PATH to an attached, audited and approved Parquet file')
    # This is only a provisional integration adapter, NOT TV2 final preprocessing.
    from mlrf.data.io import load_cleaned_parquet
    df = load_cleaned_parquet(DATA_PATH)
    if df.duplicated().any():
        raise ValueError('Duplicate rows detected. Ask TV1 to audit before splitting.')
    target = conf['dataset']['target']
    from mlrf.data.contract import load_ds04_contract
    contract = load_ds04_contract()
    numeric = [c for c in contract['numeric_columns'] if c != target]
    categorical = list(contract['categorical_columns'])
    X = df.drop(columns=[target]); y = df[target]
    if not pd.api.types.is_numeric_dtype(y):
        raise TypeError('Target price must be numeric: TV1 must clean first')
    for column in numeric:
        if not pd.api.types.is_numeric_dtype(X[column]):
            raise TypeError(f'{column} must be numeric before model training (ask TV1)')
    X = X.copy()
    for column in categorical:
        X[column] = X[column].astype('object')
    dataset_identity = {'source': conf['dataset']['repository'], 'revision': conf['dataset'].get('revision'), 'approved': True,
                        'path': str(DATA_PATH), 'rows': len(df)}
else:
    rng = np.random.default_rng(seed)
    n = 160
    X = pd.DataFrame({'year': rng.integers(2001, 2025, n),
                      'mileage': rng.integers(1000, 160000, n).astype(float),
                      'brand': rng.choice(['A','B','C'], n)})
    X.loc[10, 'mileage'] = np.nan
    y = pd.Series(5000 + (X['year'] - 2000) * 900 - X['mileage'].fillna(60000) * .05
                  + (X['brand'] == 'B') * 3500 + rng.normal(0, 1000, n), name='price')
    numeric, categorical = ['year', 'mileage'], ['brand']
    dataset_identity = {'source': 'synthetic_smoke_only', 'approved': False, 'rows': n}

assert_target_absent(X, target='price')
print('Rows:', len(X), 'numeric:', numeric, 'categorical:', categorical, dataset_identity)


In [ ]:
tr, de, te = split_indices(len(X), train=conf['split']['train'], dev=conf['split']['dev'],
                           test=conf['split']['test'], random_state=seed)
# NOTE: D13 is still provisional. Dataset with repeat vehicle/listing IDs needs group-aware split.
pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric),
    ('cat', Pipeline([('fill', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
model = Pipeline([('preprocess', pre),
                  ('regressor', build_model('custom_random_forest', {
                       'n_estimators': 12, 'max_depth': 7, 'max_features': 0.7,
                       'bootstrap': True, 'oob_score': True, 'random_state': seed}))])
model.fit(X.iloc[tr], y.iloc[tr])  # Fits preprocessing ONLY on Train.
pred_dev = model.predict(X.iloc[de])
metrics = regression_metrics(y.iloc[de], pred_dev)
print('DEV smoke metrics (not final project results):', metrics)
print('OOB:', model.named_steps['regressor'].oob_score_)
# Preserve TEST untouched; only final evaluation after model selection by TV4.


In [ ]:
root_out = Path('/kaggle/working/mlrf_runs') if Path('/kaggle/working').exists() else Path('outputs/mlrf_runs')
# Change RUN_ID for every new execution; duplicate IDs are blocked to prevent overwriting.
run_path = save_run(root_out, RUN_ID,
                    config={'model': 'custom_random_forest', 'n_estimators': 12,
                            'max_depth': 7, 'max_features': 0.7, 'seed': seed,
                            'split_status': conf['split']['status']},
                    metrics=metrics, dataset=dataset_identity,
                    git_commit=os.environ.get('GIT_COMMIT'),
                    predictions={'row_id': de, 'actual': y.iloc[de].to_numpy(), 'predicted': pred_dev})
print('Written:', run_path, '=> Save Kaggle Notebook Version with outputs after successful run')


## Next: actual dataset integration (not completed)

1. Attach approved TV1/TV2 Parquet and versioned code to Kaggle Input.
2. Confirm D10 (dataset), D12 (schema + currency + dtype), D13/D14 (split lock), TV2 preprocessing.
3. Switch from **provisional** `ColumnTransformer` to TV2's reviewed preprocessing module.
4. Set `USE_APPROVED_DATA=True`, `DATA_PATH=...`, change `RUN_ID` and review sample size/runtime before full 432k training.
5. TV4 evaluates/tunes on Dev/CV. Use Test once after config freeze. Save Kaggle output + persist run manifests/models externally.

**Security:** Never store W&B or Kaggle API tokens in notebook cells or repo.
